# When to Switch: Architecture Choice for a Workload That Moves

Three topics have chosen a retrieval architecture for a workload that was allowed to be unknown
but never to move. This notebook walks the topic that removes that last assumption.

It imports `rag_architecture_switching_hysteresis.py`, which owns every number here and in the
topic page. Nothing is recomputed in this notebook; nothing new is measured about retrieval. The
six architectures, their quality on each kind of question and both of their costs arrive through
the import chain from the three predecessor topics. The only new object is the **time axis**.

In [ ]:
import pathlib
import sys

import numpy as np

sys.path.insert(0, str(pathlib.Path.cwd() if (pathlib.Path.cwd() / "rag_architecture_switching_hysteresis.py").exists()
                       else pathlib.Path.cwd() / "notebooks" / "rag-architecture-switching-hysteresis"))
import rag_architecture_switching_hysteresis as H

print(f"arms      : {', '.join(H.ARM_NAMES)}")
print(f"windows   : {H.T_STEPS}, each observing {H.N_PER_STEP} queries")
print(f"seeds     : {H.N_SEEDS} deployments per figure")
print(f"drift     : the bridge share runs {H.DRIFT_LO} -> {H.DRIFT_HI}")

## 1. The estimate is now a stream

The predecessor observed traffic once and decided once. Here a deployment lives through
`T_STEPS` windows, and in each one it observes `N_PER_STEP` queries, forms the empirical mixture,
and decides whether to move.

Utility is affine in the workload, which is the fact that makes this affordable: each window is
an arg-max over a matrix-vector product rather than a re-measurement. `policy_streams` produces
the only two arrays the whole policy is a function of -- what the operator *sees* at each window,
and what they are *scored against*.

In [ ]:
ramp = H.drift_path("ramp")
st = H.policy_streams(ramp)
print("estimated utilities, shape", st["u_hat"].shape, "  (windows x arms)")
print("true utilities,      shape", st["u_true"].shape)
print()
print("the arm that is genuinely best at each window:")
best = np.argmax(st["u_true"], axis=1)
print("  ", "".join(str(a) for a in best))
print(f"   it changes {H.true_switch_count(ramp)}x over the whole path"
      f"  ({H.ARM_NAMES[best[0]]} -> {H.ARM_NAMES[best[-1]]})")

## 2. A rule that re-decides every window chatters

Band 0 is the predecessor's plug-in rule applied afresh at every window: switch whenever the
*estimated* best architecture changes. The workload does something once. The rule does something
a dozen times.

In [ ]:
myopic = H.over_seeds(ramp, 0.0)
print(f"the best arm really changes : {H.true_switch_count(ramp)}x")
print(f"the myopic rule switches    : {myopic['switches']:.1f}x")
print(f"ratio                       : {myopic['switches'] / max(H.true_switch_count(ramp), 1):.1f}x too often")

### The control: workloads that do not move at all

The cleanest way to show that the chatter is the *estimator* and not the workload is to remove
the workload. Every switch in the table below happens on a path with exactly zero drift, so it is
attributable to a 40-query sample and nothing else.

Deep inside a cell the myopic rule settles after its first choice. Near the boundary at 0.43 it
changes architecture on roughly half the windows of the deployment's life. This is the
predecessor's distance-to-boundary result arriving on the time axis.

In [ ]:
print(f"{'share':>6s} {'real':>5s} {'band 0':>8s} {'0.05':>7s} {'0.10':>7s} {'0.20':>7s}")
for r in H.static_chatter():
    print(f"{r['share']:6.2f} {r['true_changes']:5d} {r['band_0.0']:8.1f} "
          f"{r['band_0.05']:7.1f} {r['band_0.1']:7.1f} {r['band_0.2']:7.1f}")

## 3. The control band

Hold the incumbent until a rival beats it by more than `band`. The classical result behind this
shape -- that a fixed cost of changing state makes the optimal policy a band rather than a
threshold, and that it is optimal to do nothing inside it -- is taken as the frame and
demonstrated on this instance, not derived in general.

Watch both columns. Switching falls, which is mechanical. Regret *also* falls, which is not: the
suppressed switches were not merely expensive, they were wrong.

In [ ]:
print(f"{'band':>6s} {'switches':>9s} {'regret':>9s}")
for r in H.band_sweep():
    mark = "   <- best" if r["band"] == H.optimal_band() else ""
    print(f"{r['band']:6.3f} {r['switches']:9.1f} {r['regret']:9.5f}{mark}")

### The optimum is interior on every deployment, and its location is not asserted

An interior optimum that exists only in the mean is a weaker object than one that exists in every
run, so it is checked seed by seed. Both ends of the range lose every time.

Where the minimum sits, however, moves with the seed. This arc has been burned by a fragile
interior optimum before, so the claim the module asserts is that an interior optimum **exists**
-- never where it is.

In [ ]:
rows = H.interior_optimum_per_seed()
for r in rows:
    print(f"seed {r['seed']:2d}  best band {r['best_band']:5.3f}"
          f"  interior {str(r['interior']):>5s}"
          f"  beats myopic {str(r['beats_myopic']):>5s}"
          f"  beats widest {str(r['beats_widest']):>5s}")
print()
print(f"interior on {sum(r['interior'] for r in rows)}/{len(rows)} seeds; "
      f"the argmin lands on {sorted({r['best_band'] for r in rows})}")

## 4. Hysteresis is about noise, not about cost

We introduced the band by way of a switching cost, so the natural expectation is that the band is
a device for amortizing an expensive change -- and that making changes free should collapse it.

It does not. At a price of exactly zero the optimal band is already wide, and it stays there
across four decades of price. A band whose job was paying for switches would track the price
continuously; this one does not notice until the price is large, because its first job is
filtering an estimator.

In [ ]:
print(f"{'price':>7s} {'optimal band':>13s} {'switches':>9s} {'regret':>9s}")
for r in H.cost_sweep():
    mark = "   <- switching is FREE here" if r["cost"] == 0.0 else ""
    print(f"{r['cost']:7.3f} {r['band']:13.3f} {r['switches']:9.1f} {r['regret']:9.5f}{mark}")

## 5. Detecting is not deciding

The obvious objection: why infer drift from the decision? Monitor the workload, alarm when it
moves, switch then. The detectors for exactly this arrive unchanged from
`significance-testing-calibration` -- a two-sample KS test and PSI, over a 20-window reference
against the last 20 windows.

Pointed at the same streams the policy sees, the two instruments fail in **opposite** directions.

In [ ]:
print(f"{'path':>7s} {'real':>5s} {'myopic':>7s} {'banded':>7s} {'KS p':>8s} {'KS!':>4s} {'PSI':>7s} {'PSI!':>5s}")
for r in H.detection_vs_decision():
    print(f"{r['kind']:>7s} {r['true_changes']:5d} {r['myopic_switches']:7d} {r['banded_switches']:7d} "
          f"{r['ks_p']:8.4f} {'YES' if r['ks_alarm'] else 'no':>4s} "
          f"{r['psi']:7.3f} {'YES' if r['psi_alarm'] else 'no':>5s}")

On **static** the detectors are entirely correct -- nothing happened -- and the myopic policy
switches dozens of times regardless. Detection is right and useless, because the problem was
never that the workload moved.

On **wobble** it reverses. The best architecture genuinely changes twice, and KS sees nothing
(p = 0.83) -- also not wrong, since the path crosses the boundary, retreats, and returns near
where it started, so the first and last windows really are similar.

A detector compares two moments in time. A policy has to live through all of them. That is why
drift detection is a monitoring instrument and not a switching rule.

### The designed-path check

The drifting path is a modeling choice -- a straight line through a known boundary -- so the
verdict is re-run on a path that crosses, retreats and crosses back.

In [ ]:
for kind in ("ramp", "wobble"):
    p = H.drift_path(kind)
    m, b = H.over_seeds(p, 0.0), H.over_seeds(p, H.BAND_HEADLINE)
    print(f"{kind:>7s}: myopic {m['switches']:5.1f} switches / regret {m['regret']:.5f}"
          f"   ->   banded {b['switches']:5.1f} / {b['regret']:.5f}"
          f"   {'both improve' if b['switches'] < m['switches'] and b['regret'] < m['regret'] else 'NO'}")

## 6. The assertions

Every claim above is a test, including the ones that limit the claims: that the optimum's
location is *not* asserted, that a static path deep in a cell settles, and that the shipped
laboratory and topic page still carry the numbers this module produces today. The two drift
guards were each verified to fail on injected drift before being trusted.

In [ ]:
H._run_tests()